## Chroma DB

In [8]:
import os
from dotenv import load_dotenv
load_dotenv()


True

In [9]:
##langchain import

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_groq import ChatGroq

##Chroma DB
from langchain_community.vectorstores import Chroma

##utility

import numpy as np
from typing import List

In [10]:
sample_docs = [

    """Python is a high-level programming language used for
    web development, artificial intelligence, automation,
    and data science applications.""",

    """FAISS is a vector database library developed for
    efficient similarity search and clustering of dense vectors
    in machine learning and RAG applications.""",

    """LangChain is a framework that helps developers build
    applications powered by large language models such as
    chatbots, RAG systems, and AI agents.""",

    """Machine learning enables computers to learn patterns
    from data and make predictions without being explicitly programmed.""",

    """Groq provides ultra-fast inference for large language
    models and is commonly used to speed up AI applications."""
]

sample_docs

['Python is a high-level programming language used for\n    web development, artificial intelligence, automation,\n    and data science applications.',
 'FAISS is a vector database library developed for\n    efficient similarity search and clustering of dense vectors\n    in machine learning and RAG applications.',
 'LangChain is a framework that helps developers build\n    applications powered by large language models such as\n    chatbots, RAG systems, and AI agents.',
 'Machine learning enables computers to learn patterns\n    from data and make predictions without being explicitly programmed.',
 'Groq provides ultra-fast inference for large language\n    models and is commonly used to speed up AI applications.']

In [11]:
import tempfile

temp_dir = tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt",'w') as f:
        f.write(doc)

print(f"the created path was : {temp_dir}")

the created path was : /var/folders/1b/83xpzy2s1zz2hh1vm7mnkylm0000gn/T/tmp81u3lju0


In [12]:

for i,doc in enumerate(sample_docs):
    with open(f"doc_{i}.txt",'w') as f:
        f.write(doc)



In [13]:
from langchain_community.document_loaders import DirectoryLoader,TextLoader

loader = DirectoryLoader(
    "data",
    glob= "*.txt",
    loader_cls= TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)
documents = loader.load()

print(f"Legth of the document: {len(documents)}")
print(f"content of the page one: {documents[0].page_content[:200]}")

Legth of the document: 5
content of the page one: Machine learning enables computers to learn patterns
    from data and make predictions without being explicitly programmed.


In [14]:
documents

[Document(metadata={'source': 'data/doc_3.txt'}, page_content='Machine learning enables computers to learn patterns\n    from data and make predictions without being explicitly programmed.'),
 Document(metadata={'source': 'data/doc_2.txt'}, page_content='LangChain is a framework that helps developers build\n    applications powered by large language models such as\n    chatbots, RAG systems, and AI agents.'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='Python is a high-level programming language used for\n    web development, artificial intelligence, automation,\n    and data science applications.'),
 Document(metadata={'source': 'data/doc_1.txt'}, page_content='FAISS is a vector database library developed for\n    efficient similarity search and clustering of dense vectors\n    in machine learning and RAG applications.'),
 Document(metadata={'source': 'data/doc_4.txt'}, page_content='Groq provides ultra-fast inference for large language\n    models and is commonly u

#### Document Splitter

In [15]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len,
    separators=[" "]
)
chunks = text_splitter.split_documents(documents)

print(f"Chunk size {len(chunks)}")
print(f"content : {chunks[0].page_content[:200]}")

Chunk size 5
content : Machine learning enables computers to learn patterns
    from data and make predictions without being explicitly programmed.


In [16]:
chunks

[Document(metadata={'source': 'data/doc_3.txt'}, page_content='Machine learning enables computers to learn patterns\n    from data and make predictions without being explicitly programmed.'),
 Document(metadata={'source': 'data/doc_2.txt'}, page_content='LangChain is a framework that helps developers build\n    applications powered by large language models such as\n    chatbots, RAG systems, and AI agents.'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='Python is a high-level programming language used for\n    web development, artificial intelligence, automation,\n    and data science applications.'),
 Document(metadata={'source': 'data/doc_1.txt'}, page_content='FAISS is a vector database library developed for\n    efficient similarity search and clustering of dense vectors\n    in machine learning and RAG applications.'),
 Document(metadata={'source': 'data/doc_4.txt'}, page_content='Groq provides ultra-fast inference for large language\n    models and is commonly u

# Embedding

In [17]:
from langchain.embeddings import HuggingFaceEmbeddings
text = "what is RAG"
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
embeddings = embedding.embed_query(text)
print(embeddings)

/var/folders/1b/83xpzy2s1zz2hh1vm7mnkylm0000gn/T/ipykernel_1072/3691353408.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14150.45it/s]


[-0.07164148986339569, 0.08854860067367554, 0.0149816470220685, -0.01977272890508175, -0.08077876269817352, 0.01767987199127674, 0.024845046922564507, 0.039937008172273636, -0.014841252006590366, -0.023432757705450058, -0.01713910885155201, 0.011290241032838821, 0.028350133448839188, -0.04853471741080284, -0.03226703405380249, 0.07558141648769379, 0.046439483761787415, 0.10787294059991837, -0.015341025777161121, 0.007753883488476276, -0.07572045922279358, 0.06327486038208008, -0.010599934495985508, -0.03588681295514107, 0.05425941199064255, 0.037273455411195755, -0.012986251153051853, 0.0024181692861020565, 0.05787446349859238, -0.011427982710301876, 0.004206931684166193, 0.04149990901350975, -0.034645311534404755, -0.019974062219262123, -0.03701060637831688, 0.006360686384141445, -0.021053912118077278, 0.047576386481523514, -0.00734160328283906, 0.02255083993077278, 0.016094371676445007, 0.021969985216856003, -0.015716692432761192, -0.07138596475124359, -0.0021498841233551502, 0.03392

## Initialise the Chroma DB vector and store into the vector store

In [18]:

##create a chroma db
persist_directory = "./Chroma_DB"

##initialise the chroma db

vectorstore = Chroma.from_documents(
    documents= chunks,
    embedding=embedding,
    persist_directory=persist_directory,
    collection_name= "rag_collection"
)

print(f"vector store created with the count of {vectorstore._collection.count()} vectors")
print(f"The path of the chroma DB : {persist_directory}")


vector store created with the count of 23 vectors
The path of the chroma DB : ./Chroma_DB


## Test similarity search

In [19]:
query = "what is python?"

similarity_doc = vectorstore.similarity_search(query,k=2)
similarity_doc

[Document(metadata={'source': 'data/doc_0.txt'}, page_content='Python is a high-level programming language used for\n    web development, artificial intelligence, automation,\n    and data science applications.'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='Python is a high-level programming language used for\n    web development, artificial intelligence, automation,\n    and data science applications.')]

In [20]:
query = "what is the color of apple?"

similarity_doc = vectorstore.similarity_search(query,k=2)
similarity_doc

[Document(metadata={'topic': 'BMW Bayerische Motoren Werke AG', 'source': 'manual_adition'}, page_content="Series, 7 Series, X5, X7, and the electric i4 and iX.\nThe BMW Group also owns the Mini and Rolls-Royce brands. The company invests heavily in research and development, focusing on electric mobility, autonomous driving, and sustainable manufacturing practices.\nBMW introduced its electric vehicle sub-brand, BMW i, to accelerate the adoption of clean transportation technologies. Models such as the BMW i3 and BMW i8 were among the company's early electric and hybrid vehicles.\nThe company's slogan,"),
 Document(metadata={'topic': 'BMW Bayerische Motoren Werke AG', 'source': 'manual_adition'}, page_content="Series, 7 Series, X5, X7, and the electric i4 and iX.\nThe BMW Group also owns the Mini and Rolls-Royce brands. The company invests heavily in research and development, focusing on electric mobility, autonomous driving, and sustainable manufacturing practices.\nBMW introduced its 

In [21]:
query="what is python?"
result_score = vectorstore.similarity_search_with_score(query,k=2)
result_score

[(Document(metadata={'source': 'data/doc_0.txt'}, page_content='Python is a high-level programming language used for\n    web development, artificial intelligence, automation,\n    and data science applications.'),
  0.2798064053058624),
 (Document(metadata={'source': 'data/doc_0.txt'}, page_content='Python is a high-level programming language used for\n    web development, artificial intelligence, automation,\n    and data science applications.'),
  0.2798064053058624)]

## Initialize the LLM, RAG 

In [1]:
from langchain_groq import ChatGroq
load_dotenv()



/Users/arunrajselvarasu/Documents/RagUdemy/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'load_dotenv' is not defined

In [23]:
llm = ChatGroq(
    model_name="qwen/qwen3-32b"
)
txt = "tell about chennai"

sam_res=llm.invoke(txt)
sam_res

AIMessage(content='<think>\nOkay, I need to talk about Chennai. Let me start by recalling what I know. Chennai is a city in India, right? It\'s the capital of Tamil Nadu. I think it\'s a major city in the south of India. I remember it\'s known for its cultural heritage, maybe temples? Also, maybe something about the beach? I\'ve heard of Marina Beach being quite long.\n\nWhat else? The weather there is probably hot and humid. Does it have a monsoon season? Maybe the northeast monsoon affects it? I should check that. Oh, and Chennai is a major port city, so the port is important. It\'s also a hub for IT companies, maybe like Bangalore or Hyderabad. Companies like TCS or Wipro might have offices there.\n\nHistory-wise, was it a British colonial city? I think it was under British rule, maybe called something else before. Then maybe under the Cholas or Pallavas? There\'s a lot of historical sites there, like the Kapaleeshwarar Temple or the Fort St. George. The Marina Beach is the second-l

### Modern RAG Chain

In [24]:
from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain



In [25]:
#convert the vectore store to retriever

retriever = vectorstore.as_retriever(
    search_kwarg={'k':3}#retrieve the top 3 relavant chunks
)
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x125253a90>, search_kwargs={})

In [26]:
##create a prompt template
from langchain_core.prompts import ChatMessagePromptTemplate, ChatPromptTemplate
system_prompt = """
You are a helpful assistant.
Use only the following context to answer the user's question.
If the answer is not contained in the context, say you don't know.
Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")

])


In [27]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\nYou are a helpful assistant.\nUse only the following context to answer the user's question.\nIf the answer is not contained in the context, say you don't know.\nContext:\n{context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [28]:
#create a document chain
document_chain = create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\nYou are a helpful assistant.\nUse only the following context to answer the user's question.\nIf the answer is not contained in the context, say you don't know.\nContext:\n{context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x12b4c7e10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x12a342990>, model_name='qwen/qwen3-32b', model_kwargs

In [29]:
##create a final rag 

rag_chain = create_retrieval_chain(retriever,document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x125253a90>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\nYou are a helpful assistant.\nUse only the following context to answer the user's question.\nIf the answer is not contained in the conte

In [30]:
response = rag_chain.invoke({"input":"what is bmw"})
response

{'input': 'what is bmw',
 'context': [Document(metadata={'topic': 'BMW Bayerische Motoren Werke AG', 'source': 'manual_adition'}, page_content='BMW Bayerische Motoren Werke AG\n\n\nBMW (Bayerische Motoren Werke AG) is a German multinational manufacturer of luxury vehicles and motorcycles. The company was founded in 1916 and is headquartered in Munich, Germany.\nBMW is known for producing premium automobiles that combine performance, innovation, and luxury. Its product lineup includes sedans, SUVs, coupes, convertibles, and electric vehicles. Popular BMW models include the 3 Series, 5 Series, 7 Series, X5, X7, and the electric i4 and'),
  Document(metadata={'source': 'manual_adition', 'topic': 'BMW Bayerische Motoren Werke AG'}, page_content='BMW Bayerische Motoren Werke AG\n\n\nBMW (Bayerische Motoren Werke AG) is a German multinational manufacturer of luxury vehicles and motorcycles. The company was founded in 1916 and is headquartered in Munich, Germany.\nBMW is known for producing p

In [31]:
response['answer']

'<think>\nOkay, the user is asking "what is bmw". Let me check the context provided.\n\nThe context starts with BMW standing for Bayerische Motoren Werke AG. It\'s a German multinational manufacturer of luxury vehicles and motorcycles, founded in 1916 and based in Munich. They make premium cars with performance, innovation, and luxury. The product line includes sedans, SUVs, coupes, convertibles, and electric vehicles. Popular models are mentioned like 3 Series, 5 Series, 7 Series, X5, X7, and electric models i4 and iX. They also own Mini and Rolls-Royce. They focus on electric mobility, autonomous driving, and sustainable practices. The electric sub-brand is BMW i, with models i3 and i8. Their slogan is mentioned but cut off.\n\nSince the user just wants to know what BMW is, I should summarize the key points: the company\'s full name, origin, founding year, headquarters, product range, ownership of other brands, and their focus areas. Make sure to mention both traditional and electric

## create a RAG Chain with a help of LCEL (Langchain Expression Language)

In [32]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel


In [33]:
custom_prompt = ChatPromptTemplate.from_template(
"""
You are an expert AI assistant.
Use only the provided context to answer the user's question.
Guidelines:

- Be accurate and concise.
- If the answer is not present in the context, say:
  "I don't know based on the provided context."
- Do not make up information.
- Explain concepts clearly when enough context is available.
- Use bullet points when appropriate.
Context:
{context}
Question:
{question}
Answer:
"""
)
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nYou are an expert AI assistant.\nUse only the provided context to answer the user\'s question.\nGuidelines:\n\n- Be accurate and concise.\n- If the answer is not present in the context, say:\n  "I don\'t know based on the provided context."\n- Do not make up information.\n- Explain concepts clearly when enough context is available.\n- Use bullet points when appropriate.\nContext:\n{context}\nQuestion:\n{question}\nAnswer:\n'), additional_kwargs={})])

In [34]:
#format the docs

def fromat_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [35]:
##Build the rag chain using the LCEL
rag_chain_lcel = (
    {
        "context": retriever | fromat_docs,
        "question": RunnablePassthrough()
    }
    | custom_prompt
    | llm
    | StrOutputParser()
)

In [36]:
rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x125253a90>, search_kwargs={})
           | RunnableLambda(fromat_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nYou are an expert AI assistant.\nUse only the provided context to answer the user\'s question.\nGuidelines:\n\n- Be accurate and concise.\n- If the answer is not present in the context, say:\n  "I don\'t know based on the provided context."\n- Do not make up information.\n- Explain concepts clearly when enough context is available.\n- Use bullet points when appropriate.\nContext:\n{context}\nQuestion:\n{question}\nAnswer:\n'), additional_kwargs={})])
| ChatGroq(client=<groq.resources.chat.

In [37]:
rag_chain_lcel.invoke("What is chennai")

'<think>\nOkay, the user is asking "What is chennai". Let me check the provided context.\n\nLooking at the context, the first part is about machine learning and how it helps computers learn patterns from data. Then there\'s a repetition of that same text twice. After that, there\'s a paragraph about Python being a high-level programming language used in web development, AI, automation, and data science.\n\nNowhere in this context is there any mention of Chennai. The context is focused on machine learning and Python. The user\'s question is about a city, which isn\'t covered in the given information. Since I can\'t use any external knowledge and have to rely solely on the provided context, I need to respond that I don\'t know the answer based on the context given. There\'s no information here about Chennai\'s location, population, or anything else related to it. So the correct response is to state that the context doesn\'t provide that information.\n</think>\n\nI don\'t know based on th

## Add the document Existing VectorStore

In [38]:
vectorstore

In [39]:
new_document = """
BMW Bayerische Motoren Werke AG


BMW (Bayerische Motoren Werke AG) is a German multinational manufacturer of luxury vehicles and motorcycles. The company was founded in 1916 and is headquartered in Munich, Germany.
BMW is known for producing premium automobiles that combine performance, innovation, and luxury. Its product lineup includes sedans, SUVs, coupes, convertibles, and electric vehicles. Popular BMW models include the 3 Series, 5 Series, 7 Series, X5, X7, and the electric i4 and iX.
The BMW Group also owns the Mini and Rolls-Royce brands. The company invests heavily in research and development, focusing on electric mobility, autonomous driving, and sustainable manufacturing practices.
BMW introduced its electric vehicle sub-brand, BMW i, to accelerate the adoption of clean transportation technologies. Models such as the BMW i3 and BMW i8 were among the company's early electric and hybrid vehicles.
The company's slogan, "The Ultimate Driving Machine," reflects its emphasis on driving dynamics and engineering excellence. BMW operates manufacturing facilities in multiple countries and sells vehicles in markets worldwide.
In recent years, BMW has expanded its electric vehicle portfolio and committed to reducing carbon emissions throughout its supply chain. The company aims to balance luxury, performance, and environmental responsibility while maintaining its position as one of the leading premium automotive manufacturers globally."""

In [40]:
vectorstore

In [41]:
new_document

'\nBMW Bayerische Motoren Werke AG\n\n\nBMW (Bayerische Motoren Werke AG) is a German multinational manufacturer of luxury vehicles and motorcycles. The company was founded in 1916 and is headquartered in Munich, Germany.\nBMW is known for producing premium automobiles that combine performance, innovation, and luxury. Its product lineup includes sedans, SUVs, coupes, convertibles, and electric vehicles. Popular BMW models include the 3 Series, 5 Series, 7 Series, X5, X7, and the electric i4 and iX.\nThe BMW Group also owns the Mini and Rolls-Royce brands. The company invests heavily in research and development, focusing on electric mobility, autonomous driving, and sustainable manufacturing practices.\nBMW introduced its electric vehicle sub-brand, BMW i, to accelerate the adoption of clean transportation technologies. Models such as the BMW i3 and BMW i8 were among the company\'s early electric and hybrid vehicles.\nThe company\'s slogan, "The Ultimate Driving Machine," reflects its e

In [42]:
chunks

[Document(metadata={'source': 'data/doc_3.txt'}, page_content='Machine learning enables computers to learn patterns\n    from data and make predictions without being explicitly programmed.'),
 Document(metadata={'source': 'data/doc_2.txt'}, page_content='LangChain is a framework that helps developers build\n    applications powered by large language models such as\n    chatbots, RAG systems, and AI agents.'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='Python is a high-level programming language used for\n    web development, artificial intelligence, automation,\n    and data science applications.'),
 Document(metadata={'source': 'data/doc_1.txt'}, page_content='FAISS is a vector database library developed for\n    efficient similarity search and clustering of dense vectors\n    in machine learning and RAG applications.'),
 Document(metadata={'source': 'data/doc_4.txt'}, page_content='Groq provides ultra-fast inference for large language\n    models and is commonly u

In [43]:
from langchain_core.documents import Document

new_doc = Document(
    page_content=new_document,
    metadata={"source": "manual_adition", "topic": "BMW Bayerische Motoren Werke AG"}
)

In [44]:
new_doc

Document(metadata={'source': 'manual_adition', 'topic': 'BMW Bayerische Motoren Werke AG'}, page_content='\nBMW Bayerische Motoren Werke AG\n\n\nBMW (Bayerische Motoren Werke AG) is a German multinational manufacturer of luxury vehicles and motorcycles. The company was founded in 1916 and is headquartered in Munich, Germany.\nBMW is known for producing premium automobiles that combine performance, innovation, and luxury. Its product lineup includes sedans, SUVs, coupes, convertibles, and electric vehicles. Popular BMW models include the 3 Series, 5 Series, 7 Series, X5, X7, and the electric i4 and iX.\nThe BMW Group also owns the Mini and Rolls-Royce brands. The company invests heavily in research and development, focusing on electric mobility, autonomous driving, and sustainable manufacturing practices.\nBMW introduced its electric vehicle sub-brand, BMW i, to accelerate the adoption of clean transportation technologies. Models such as the BMW i3 and BMW i8 were among the company\'s e

In [45]:
#split the documents
new_chunks = text_splitter.split_documents([new_doc])
new_chunks

[Document(metadata={'source': 'manual_adition', 'topic': 'BMW Bayerische Motoren Werke AG'}, page_content='BMW Bayerische Motoren Werke AG\n\n\nBMW (Bayerische Motoren Werke AG) is a German multinational manufacturer of luxury vehicles and motorcycles. The company was founded in 1916 and is headquartered in Munich, Germany.\nBMW is known for producing premium automobiles that combine performance, innovation, and luxury. Its product lineup includes sedans, SUVs, coupes, convertibles, and electric vehicles. Popular BMW models include the 3 Series, 5 Series, 7 Series, X5, X7, and the electric i4 and'),
 Document(metadata={'source': 'manual_adition', 'topic': 'BMW Bayerische Motoren Werke AG'}, page_content="Series, 7 Series, X5, X7, and the electric i4 and iX.\nThe BMW Group also owns the Mini and Rolls-Royce brands. The company invests heavily in research and development, focusing on electric mobility, autonomous driving, and sustainable manufacturing practices.\nBMW introduced its elect

In [46]:
#add new documents to vectorstore

vectorstore.add_documents(new_chunks)

['d888cb9a-9269-4a48-a02d-04055332d52a',
 '1aaccd2c-79c0-4bb7-bc2c-6f0ee14f1e30',
 '52ecdde1-0430-47eb-89c3-7fdf7c102a0e',
 'a51eceba-442a-4462-87f5-4d8c71a89daf']

In [47]:
print(f"Added Chunks : {len(new_chunks)} new chunk to the vector store")

Added Chunks : 4 new chunk to the vector store


In [48]:
print(f"Total vectors now: {vectorstore._collection.count()}")

Total vectors now: 27


In [49]:
##query with the updated vector
new_question = "What is BMW, what are the version ?"
result = rag_chain_lcel.invoke(new_document)
result

'<think>\nOkay, let\'s see. The user asked about BMW Bayerische Motoren Werke AG. The context provided has several sections about BMW.\n\nFirst, I need to check the question. The user\'s question is just the same text repeated multiple times, but maybe they intended to ask something specific about BMW. Since the actual question isn\'t clear, but looking at the history, maybe they want a summary or specific info about BMW.\n\nLooking at the context, it mentions BMW is a German multinational company founded in 1916, based in Munich. They make luxury vehicles and motorcycles. Product lines include sedans, SUVs, etc., with models like 3 Series, X5, and electric cars like i4 and iX. They own Mini and Rolls-Royce. Their slogan is "The Ultimate Driving Machine." They focus on electric vehicles under BMW i, with models i3 and i8. Also, R&D in electric mobility, autonomous driving, sustainable practices. They have global operations.\n\nThe user\'s answer should be based on this. Since the quest

## In RAG Add Converstional Memory

In [71]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

In [72]:

contextualize_q_system_prompt = """
Given the chat history and the latest user question,
rewrite the latest question into a standalone question
that can be understood without the chat history.

Rules:
- Preserve the original meaning.
- Replace pronouns and references (e.g., "it", "they", "that company")
  with the actual entities from the chat history.
- Do not answer the question.
- Do not add new information.
- Return only the rewritten standalone question.

If the question is already standalone, return it unchanged.
"""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human","{input}")
]
)

In [73]:
##create a history aware retriever

history_aware_retriever = create_history_aware_retriever(
    llm,retriever,contextualize_q_prompt
)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x125253a90>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMes

In [74]:
qa_system_prompt = """
You are a helpful AI assistant.

Use the retrieved context to answer the user's question.

Instructions:
- Answer only using the information provided in the context.
- If the answer cannot be found in the context, say:
  "I don't know based on the provided context."
- Do not make up facts or use external knowledge.
- Be clear, concise, and accurate.
- If appropriate, format the answer using bullet points.

Context:
{context}
"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human","{input}")
])

question_answer_chain = create_stuff_documents_chain(llm,qa_prompt)
question_answer_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['chat_history', 'context', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[l

In [75]:
conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain
)
print("converstional rag was created !!")

converstional rag was created !!


In [76]:
#first question

chat_history=[]

result = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What is groq?"
})


print("Question: What is groq?")
print(result['answer'])

Question: What is groq?
<think>
Okay, the user is asking, "What is Groq?" Let me look at the context provided.

The context mentions that Groq provides ultra-fast inference for large language models and is commonly used to speed up AI applications. This information is repeated three times. There's also information about LangChain, but the user is specifically asking about Groq.

So, the answer should focus on Groq's role in providing fast inference for LLMs and its use in AI applications. Since the context doesn't mention anything else about Groq, like its founding year, competitors, or specific use cases beyond speeding up AI apps, I need to stick to what's given. The user might be looking for a concise definition, so I'll present the key points clearly. No need to include LangChain info here. Just summarize the Groq part from the context.
</think>

Groq provides ultra-fast inference for large language models and is commonly used to speed up AI applications.


In [77]:
chat_history

[]

In [78]:
chat_history.extend([
    HumanMessage(content="what is groq?"),
    AIMessage(content=result['answer'])
]
)

In [79]:
chat_history

[HumanMessage(content='what is groq?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='<think>\nOkay, the user is asking, "What is Groq?" Let me look at the context provided.\n\nThe context mentions that Groq provides ultra-fast inference for large language models and is commonly used to speed up AI applications. This information is repeated three times. There\'s also information about LangChain, but the user is specifically asking about Groq.\n\nSo, the answer should focus on Groq\'s role in providing fast inference for LLMs and its use in AI applications. Since the context doesn\'t mention anything else about Groq, like its founding year, competitors, or specific use cases beyond speeding up AI apps, I need to stick to what\'s given. The user might be looking for a concise definition, so I\'ll present the key points clearly. No need to include LangChain info here. Just summarize the Groq part from the context.\n</think>\n\nGroq provides ultra-fast inference for large

In [ ]:
##follow up
result1 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What are the diiferent types of models?"
})
result1

{'chat_history': [HumanMessage(content='what is groq?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='<think>\nOkay, the user is asking "what is bmw". Let me check the context provided.\n\nThe context starts with BMW standing for Bayerische Motoren Werke AG. It\'s a German multinational manufacturer of luxury vehicles and motorcycles, founded in 1916 and based in Munich. They make premium cars with performance, innovation, and luxury. The product line includes sedans, SUVs, coupes, convertibles, and electric vehicles. Popular models are mentioned like 3 Series, 5 Series, 7 Series, X5, X7, and electric models i4 and iX. They also own Mini and Rolls-Royce. They focus on electric mobility, autonomous driving, and sustainable practices. The electric sub-brand is BMW i, with models i3 and i8. Their slogan is mentioned but cut off.\n\nSince the user just wants to know what BMW is, I should summarize the key points: the company\'s full name, origin, founding year, headquar

In [70]:
result1['answer']

"<think>\nOkay, the user is asking about the different types of models. Let me check the context they provided.\n\nLooking back, the context mentions Groq and LangChain. Groq is about ultra-fast inference for large language models, and LangChain is a framework for building apps with LLMs like chatbots, RAG systems, and AI agents. But the user is asking about types of models in general. \n\nWait, the context doesn't list different model types. It only talks about large language models in the case of Groq and LangChain's applications. There's no mention of other model categories like computer vision models, reinforcement learning models, or others. \n\nThe user's question is broader than the provided context. Since I can't use external knowledge, I need to say I don't know based on the context. But maybe I should check again. The context repeats Groq's info three times, but no other model types. So yes, the answer isn't here. I should respond that I can't provide the answer with the give